In [ ]:
import openslide
import matplotlib.pyplot as plt
import numpy as np
import glob

In [ ]:
from transformers import LlavaNextProcessor, LlavaNextForConditionalGeneration
import torch
from PIL import Image
import requests

processor = LlavaNextProcessor.from_pretrained("llava-hf/llama3-llava-next-8b-hf")
model = LlavaNextForConditionalGeneration.from_pretrained("llava-hf/llama3-llava-next-8b-hf", torch_dtype=torch.float16).cuda()#, device_map="auto") 

In [ ]:
slides = glob.glob("/nfs/turbo/umms-tocho/data/he.mlins_scan.svs.raw/20241108/*.svs")

In [ ]:
len(slides)

In [ ]:
fname_map = {}
for slide_fn in slides:
    slide = openslide.OpenSlide(slide_fn)
    label_image = slide.associated_images["label"]

    prompt = "Perform OCR on this image. The result is of the format M\\d+x\\d+. Return only the text on the image, without describing it."
    conversation = [
        {

          "role": "user",
          "content": [
              {"type": "text", "text": prompt},
              {"type": "image"},
            ],
        },
    ]
    prompt = processor.apply_chat_template(conversation, add_generation_prompt=True)
    inputs = processor(images=label_image, text=prompt, return_tensors="pt").to(model.device)
    output = model.generate(**inputs, max_new_tokens=100)
    out_text = processor.decode(output[0], skip_special_tokens=True).split("\n")[-1]
    fname_map[slide_fn] = out_text
    
    fig, axs = plt.subplots(1,2,  width_ratios=[1, 3])
    axs[0].imshow(slide.associated_images["label"])
    axs[0].set_title(out_text)
    axs[1].imshow(slide.associated_images["macro"])
    

In [ ]:
results = pd.DataFrame.from_dict(fname_map, orient='index').reset_index()
results.columns=["path", "mx_code"]
results["mx_code"] = results["mx_code"].str.lower()


def get_mcode(x):
    delimiters = [" ", "x", "k", "+"]
 
    for delimiter in delimiters:
        x = " ".join(x.split(delimiter))

    return x.split()[0]

def get_xcode(x):
    delimiters = [" ", "x", "k", "+"]
 
    for delimiter in delimiters:
        x = x.strip(delimiter)
        x = " ".join(x.split(delimiter))

    return "x"+ x.strip().split()[-1]

results["m_code"] = results["mx_code"].apply(get_mcode)
results["x_code"] = results["mx_code"].apply(get_xcode)

results = results.sort_values(by=["m_code", "x_code"])
results.to_csv("20241108v2.csv", index=False)

In [ ]:
results